In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
task_1_result_path = "../artifacts/pos_tagging_results.csv"
task_2_result_path = "../artifacts/pos_tagging_results.csv"

task_1_name = "POS Tagging 1"
task_2_name = "POS Tagging 2"

In [3]:
task_1_results = pd.read_csv(task_1_result_path)
task_2_results = pd.read_csv(task_2_result_path)

In [4]:
color_map = {"Traditional": "#636EFA", "BiLSTM": "#EF553B", "Transformer": "#00CC96"}

for df, task in [(task_1_results, task_1_name), (task_2_results, task_2_name)]:
    df["task"] = task
    df["family"] = df["name"].apply(
        lambda x: "Traditional"
        if any(t in x for t in ["hmm", "crf"])
        else "BiLSTM"
        if "bilstm" in x
        else "Transformer"
    )
combined_results = pd.concat([task_1_results, task_2_results])

In [15]:
fig_pareto_grid = make_subplots(
    rows=1, cols=2, subplot_titles=(task_1_name, task_2_name), shared_yaxes=True
)

for i, df in enumerate([task_1_results, task_2_results]):
    for family in df["family"].unique():
        sub = df[df["family"] == family]
        fig_pareto_grid.add_trace(
            go.Scatter(
                x=sub["p50_latency_ms"],
                y=sub["macro_f1"],
                name=family,
                mode="markers+text",
                textposition="top center",
                marker=dict(size=12, color=color_map[family]),
                legendgroup=family,
                showlegend=(i == 0),
                hovertext=sub["name"],
            ),
            row=1,
            col=i + 1,
        )

fig_pareto_grid.update_layout(
    title_text="F1(Macro) vs. Latency, Pareto Frontier",
    title_x=0.5,
    template="plotly_white",
    height=500,
)
fig_pareto_grid.update_xaxes(title_text="p50 Latency (log ms)", type="log")
fig_pareto_grid.update_yaxes(title_text="f1(macro)")

fig_pareto_grid.show()

In [ ]:
fig_throughput_grid = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(task_1_name, task_2_name),
    shared_yaxes=False,
    horizontal_spacing=0.25,
)

for i, df in enumerate([task_1_results, task_2_results]):
    sorted_df = df.sort_values("throughput_seq_per_sec", ascending=True)

    # Create hover text with additional metrics
    hover_text = [
        f"<b>{name}</b><br>"
        + f"Throughput: {throughput:.1f} seq/s<br>"
        + f"F1 (Macro): {f1:.3f}<br>"
        + f"p50 Latency: {latency:.2f} ms<br>"
        + f"Memory: {mem:.1f} MB"
        for name, throughput, f1, latency, mem in zip(
            sorted_df["name"],
            sorted_df["throughput_seq_per_sec"],
            sorted_df["macro_f1"],
            sorted_df["p50_latency_ms"],
            sorted_df["peak_memory_mb"],
        )
    ]

    fig_throughput_grid.add_trace(
        go.Bar(
            x=sorted_df["throughput_seq_per_sec"],
            y=sorted_df["name"],
            orientation="h",
            marker=dict(
                color=[color_map[f] for f in sorted_df["family"]],
                line=dict(color="rgba(0,0,0,0.3)", width=1),
            ),
            text=[f"{val:.0f}" for val in sorted_df["throughput_seq_per_sec"]],
            textposition="outside",
            textfont=dict(size=10),
            hovertext=hover_text,
            hoverinfo="text",
            showlegend=False,
            legendgroup=f"data_{i}",
        ),
        row=1,
        col=i + 1,
    )

# Add legend manually using dummy traces (only once)
for family, color in color_map.items():
    fig_throughput_grid.add_trace(
        go.Scatter(
            x=[None],
            y=[None],
            mode="markers",
            marker=dict(color=color, size=10),
            name=family,
            showlegend=True,
            legendgroup=family,
        )
    )

fig_throughput_grid.update_layout(
    title_text="Throughput Comparison: Sequences per Second",
    title_x=0.5,
    template="plotly_white",
    height=800,
    showlegend=True,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5),
    margin=dict(l=200, r=100, t=120, b=80),
)

fig_throughput_grid.update_xaxes(
    title_text="Throughput (log sequences/sec)", type="log", gridcolor="lightgray"
)
fig_throughput_grid.update_yaxes(title_text="", tickfont=dict(size=10))

fig_throughput_grid.show()

In [18]:
rep_models = [
    "crf",
    "bilstm-batch-128-smart-batching",
    "transformer-batch-128-smart-batching",
]
slope_df = combined_results[combined_results["name"].isin(rep_models)]

fig_slope = go.Figure()
for model in rep_models:
    model_data = slope_df[slope_df["name"] == model]
    fig_slope.add_trace(
        go.Scatter(
            x=model_data["task"],
            y=model_data["macro_f1"],
            mode="lines+markers+text",
            name=model,
            text=model_data["macro_f1"].round(3),
            textposition="top center",
        )
    )
fig_slope.update_layout(
    title="Direct Comparison: F1 Degradation by Task Complexity",
    yaxis_title="Macro F1 Score",
    template="plotly_white",
    title_x=0.5,
    width=800,
    height=500,
)
fig_slope.show()

In [ ]:
combined_results["efficiency"] = combined_results["macro_f1"] / np.log1p(
    combined_results["p50_latency_ms"]
)

efficiency_sorted = combined_results.sort_values("efficiency", ascending=True)

fig_efficiency = px.bar(
    efficiency_sorted,
    y="name",
    x="efficiency",
    color="task",
    barmode="group",
    title="Efficiency Score: F1 Score per Log-Latency Unit ",
    labels={"efficiency": "Efficiency Score", "name": "Model"},
    template="plotly_white",
    height=600,
    orientation="h",
)

fig_efficiency.update_layout(
    xaxis_title="Efficiency Score (F1 / log(latency))",
    yaxis_title="",
    title_x=0.5,
    legend=dict(
        orientation="v", yanchor="top", y=1, xanchor="left", x=1.02, title="Task"
    ),
    margin=dict(l=250, r=150, t=80, b=60),
)

fig_efficiency.update_yaxes(tickfont=dict(size=10))

fig_efficiency.show()